In [7]:
import pandas as pd
import sqlite3

In [8]:
df = pd.read_csv("panel_long_quarterly_with_acc_ca.csv")

df.head()

,country,date,variable,value
0,AT,2000Q1,spread,0.2431204278812968
1,AT,2000Q2,spread,0.2899663299663269
2,AT,2000Q3,spread,0.3221471359558308
3,AT,2000Q4,spread,0.3105314878104375
4,AT,2001Q1,spread,0.3196303030303031


In [9]:
conn = sqlite3.connect("project.db")

df.to_sql("panel_data", conn, if_exists="replace", index=False)

print("Data successfully loaded into SQL table")

Data successfully loaded into SQL table


In [10]:
# Query 1

query = """
SELECT *
FROM panel_data
LIMIT 20
"""

pd.read_sql(query, conn)



,country,date,variable,value
0,AT,2000Q1,spread,0.2431204278812968
1,AT,2000Q2,spread,0.2899663299663269
2,AT,2000Q3,spread,0.3221471359558308
3,AT,2000Q4,spread,0.3105314878104375
4,AT,2001Q1,spread,0.3196303030303031
5,AT,2001Q2,spread,0.3008623604465699
6,AT,2001Q3,spread,0.2726259552042158
7,AT,2001Q4,spread,0.2411083856467497
8,AT,2002Q1,spread,0.1993242424242431
9,AT,2002Q2,spread,0.1976115440115426


WHAT: This query displays the first 20 rows of the dataset.

HOW: The SELECT statement retrieves all columns from the panel_data table, and LIMIT restricts the output to the first 20 observations.

WHY:This helps verify that the dataset loaded correctly and allows us to inspect the structure of the data before performing analysis.

In [11]:
# Query 2


query = """
SELECT country,
       COUNT(*) AS observations
FROM panel_data
GROUP BY country
ORDER BY observations DESC
"""

pd.read_sql(query, conn)



,country,observations
0,PT,800
1,LU,800
2,IT,800
3,FR,800
4,FI,800
5,ES,800
6,DE,800
7,AT,800
8,IE,784
9,GR,784


WHAT: This query counts how many observations exist for each country.

HOW: The COUNT() function counts rows, and GROUP BY aggregates the results for each country, and ORDER BY sorts the data by observations in descending order.

WHY: This helps determine how balanced the panel dataset is and whether some countries have more data observations than others.

In [12]:
# Query 3

query = """
SELECT country,
       AVG(value) AS avg_spread
FROM panel_data
WHERE variable = 'spread'
GROUP BY country
ORDER BY avg_spread DESC
"""

pd.read_sql(query, conn)


,country,avg_spread
0,GR,3.924514
1,PT,1.760314
2,IT,1.336504
3,IE,1.070474
4,ES,1.026203
5,BE,0.488914
6,FR,0.363014
7,AT,0.348723
8,FI,0.275042
9,NL,0.204464


WHAT: This query calculates the average sovereign bond spread for each country.

HOW: The query filters rows where the variable equals 'spread'(WHERE). Then it calculates the average using AVG() and groups results (GROUP BY) by country in descending order (ORDER).

WHY:This helps identify which countries tend to have higher borrowing

In [13]:
# Query 4


query = """
SELECT s.country,
       s.date,
       s.value AS spread,
       d.value AS debt_gdp
FROM panel_data s
JOIN panel_data d
ON s.country = d.country
AND s.date = d.date
WHERE s.variable = 'spread'
AND d.variable = 'debt_gdp'
LIMIT 50
"""

pd.read_sql(query, conn)



,country,date,spread,debt_gdp
0,AT,2000Q1,0.2431204278812968,70.8
1,AT,2000Q2,0.2899663299663269,71.4
2,AT,2000Q3,0.3221471359558308,72.0
3,AT,2000Q4,0.3105314878104375,66.6
4,AT,2001Q1,0.3196303030303031,70.4
5,AT,2001Q2,0.3008623604465699,71.1
6,AT,2001Q3,0.2726259552042158,71.0
7,AT,2001Q4,0.2411083856467497,67.2
8,AT,2002Q1,0.1993242424242431,71.5
9,AT,2002Q2,0.1976115440115426,70.9


WHAT:This query combines spread data with debt-to-GDP values for the same country and quarter.

HOW:
SELECT s.country
Selects the country identifier from the dataset.

s.date
Selects the quarter (time period) of the observation.

s.value AS spread
Retrieves the value column from rows where the variable is 'spread'
 and renames the column as "spread" in the output.

d.value AS debt_gdp
Retrieves the value column from rows where the variable is 'debt_gdp'
and renames the column as "debt_gdp".

FROM panel_data s
Specifies the dataset to use and assigns it the alias "s".
The alias "s" represents rows that contain spread observations.

JOIN panel_data d
Joins the dataset with itself using the alias "d".
This alias represents rows containing debt-to-GDP observations.

ON s.country = d.country AND s.date = d.date
Defines the join condition. Rows are matched when the country
and date are the same, ensuring the variables correspond to
the same country and quarter.

WHERE s.variable = 'spread'
Filters rows so that alias "s" only includes spread observations.

AND d.variable = 'debt_gdp'
Filters rows so that alias "d" only includes debt-to-GDP observations.

LIMIT 50
Restricts the output to the first 50 rows to keep the results manageable
when displaying them in the notebook.

WHY:
Because the dataset is stored in long format, variables are stored in rows instead of columns. This join allows us to analyze therelationship between sovereign spreads and government debt levels.

In [14]:
# Query 5

query = """
SELECT s.country,
       s.date,
       s.value AS spread,
       d.value AS debt_gdp,
       g.value AS gdp_growth
FROM panel_data s
JOIN panel_data d
ON s.country = d.country AND s.date = d.date
JOIN panel_data g
ON s.country = g.country AND s.date = g.date
WHERE s.variable = 'spread'
AND d.variable = 'debt_gdp'
AND g.variable = 'gdp_growth'
LIMIT 50
"""

pd.read_sql(query, conn)



,country,date,spread,debt_gdp,gdp_growth
0,AT,2000Q1,0.2431204278812968,70.8,n.a.
1,AT,2000Q2,0.2899663299663269,71.4,1.13528710439396
2,AT,2000Q3,0.3221471359558308,72.0,1.39878571946425
3,AT,2000Q4,0.3105314878104375,66.6,0.67558462301597
4,AT,2001Q1,0.3196303030303031,70.4,-0.30145287373084
5,AT,2001Q2,0.3008623604465699,71.1,-0.28603936750446
6,AT,2001Q3,0.2726259552042158,71.0,0.40272768532325
7,AT,2001Q4,0.2411083856467497,67.2,0.2353083709868599
8,AT,2002Q1,0.1993242424242431,71.5,1.10061321513317
9,AT,2002Q2,0.1976115440115426,70.9,0.26219441736436


WHAT:
This query combines three macroeconomic variables:
spread, debt-to-GDP, and GDP growth.

HOW:
SELECT s.country
Selects the country identifier from the dataset.

s.date
Selects the quarter of the observation.

s.value AS spread
Retrieves the value column for rows where the variable is 'spread'
and renames it as "spread".

d.value AS debt_gdp
 Retrieves the value column for rows where the variable is 'debt_gdp'
and renames it as "debt_gdp".

g.value AS gdp_growth
Retrieves the value column for rows where the variable is 'gdp_growth'
and renames it as "gdp_growth".

FROM panel_data s
Defines the main table and assigns it the alias "s".
This alias represents rows where the variable is spread.

JOIN panel_data d
Joins the table to itself using the alias "d",
which represents rows containing debt-to-GDP values.

ON s.country = d.country AND s.date = d.date
Matches rows where the country and date are the same,
ensuring spread and debt values correspond to the same observation.

JOIN panel_data g
Joins the table a third time using alias "g",
which represents rows containing GDP growth values.

ON s.country = g.country AND s.date = g.date
Matches GDP growth observations with the same country and quarter.

WHERE s.variable = 'spread'
Filters the rows for alias "s" so that only spread observations are used.

AND d.variable = 'debt_gdp'
Filters rows for alias "d" so that only debt-to-GDP observations are included.

AND g.variable = 'gdp_growth'
Filters rows for alias "g" so that only GDP growth observations are included.
LIMIT limits the number of outputs to 50.


WHY:
This allows us to examine how economic growth and government debt may influence sovereign borrowing costs.

In [ ]:
# Query 6

query = """
SELECT s.country,
       AVG(s.value) AS avg_spread,
       AVG(d.value) AS avg_debt
FROM panel_data s
JOIN panel_data d
ON s.country = d.country
AND s.date = d.date
WHERE s.variable = 'spread'
AND d.variable = 'debt_gdp'
GROUP BY s.country
ORDER BY avg_debt DESC
"""

pd.read_sql(query, conn)



WHAT:
This query calculates the average sovereign spread and the average debt-to-GDP ratio for each country.

HOW:
The query joins spread observations with debt-to-GDP observations
for the same country and quarter. Then it calculates averages and
groups the results by country.

SELECT s.country
Selects the country identifier.

AVG(s.value) AS avg_spread
Calculates the average value of spreads for each country
and renames the column as "avg_spread".

AVG(d.value) AS avg_debt
Calculates the average debt-to-GDP ratio for each country
and renames the column as "avg_debt".

FROM panel_data s
Defines the dataset and assigns the alias "s".
This alias represents spread observations.

JOIN panel_data d
Joins the dataset with itself using alias "d".
Alias "d" represents debt-to-GDP observations.

ON s.country = d.country AND s.date = d.date
Matches rows with the same country and date to ensure the
spread and debt values belong to the same observation.

WHERE s.variable = 'spread'
 Filters rows so alias "s" only contains spread observations.

AND d.variable = 'debt_gdp'
Filters rows so alias "d" only contains debt-to-GDP observations.

GROUP BY s.country
Groups the results by country so averages are calculated
separately for each country.

ORDER BY avg_debt DESC
Sorts the output so countries with the highest average
debt-to-GDP ratios appear first.

WHY:This query helps compare countries based on their average debt levels and borrowing costs.

In [16]:
# Query 7


query = """
SELECT country,
       AVG(value) AS avg_spread
FROM panel_data
WHERE variable = 'spread'
GROUP BY country
HAVING AVG(value) >
       (SELECT AVG(value)
        FROM panel_data
        WHERE variable = 'spread')
"""

pd.read_sql(query, conn)



,country,avg_spread
0,ES,1.026203
1,GR,3.924514
2,IE,1.070474
3,IT,1.336504
4,PT,1.760314


WHAT:
This query identifies countries whose average spread is higher
than the overall average spread in the dataset.

HOW:
The query calculates each country's average spread and compares it
to the global average spread using a subquery.

SELECT country
Selects the country identifier.

AVG(value) AS avg_spread
Calculates the average spread for each country.

FROM panel_data
Specifies the dataset being used.

WHERE variable = 'spread'
Filters rows so that only spread observations are included.

GROUP BY country
Groups rows by country so the average spread can be calculated
separately for each country.

HAVING AVG(value) >
Filters the grouped results so only countries with spreads
above the global average remain.

(SELECT AVG(value)
This subquery calculates the overall average spread.

FROM panel_data
Specifies the dataset used in the subquery.

WHERE variable = 'spread')
Filters the subquery to only use spread observations.

WHY:
This query helps identify countries that consistently have
higher borrowing costs than the dataset average.

In [17]:
# Query 8


query = """
SELECT country,
       date,
       value AS debt_gdp
FROM panel_data
WHERE variable = 'debt_gdp'
AND value >
      (SELECT AVG(value)
       FROM panel_data
       WHERE variable = 'debt_gdp')
ORDER BY value DESC
LIMIT 50
"""

pd.read_sql(query, conn)


,country,date,debt_gdp
0,BE,2009Q4,99.9
1,PT,2010Q4,99.9
2,ES,2018Q4,99.8
3,BE,2005Q2,99.7
4,FR,2016Q2,99.7
5,ES,2019Q3,99.6
6,FR,2018Q3,99.6
7,FR,2019Q1,99.5
8,FR,2018Q1,99.4
9,FR,2018Q2,99.4


WHAT:
This query identifies country-quarter observations where
the debt-to-GDP ratio is above the overall dataset average.

HOW:
A subquery calculates the average debt-to-GDP ratio, and
the outer query returns observations above that value.

SELECT country
Selects the country identifier.

date
Selects the quarter of the observation.

value AS debt_gdp
Retrieves the value column and renames it as "debt_gdp".

FROM panel_data
Specifies the dataset being used.

WHERE variable = 'debt_gdp'
Filters rows so that only debt-to-GDP observations are included.

AND value >
Filters rows so only values above the dataset average are returned.

(SELECT AVG(value)
Subquery that calculates the average debt-to-GDP ratio.

FROM panel_data
Specifies the dataset for the subquery.

WHERE variable = 'debt_gdp')
Ensures only debt observations are used in the average calculation.

ORDER BY value DESC
Sorts the results from highest to lowest debt level.

LIMIT 50
Restricts the output to the first 50 rows.

WHY:
This query helps identify periods when government debt
reached unusually high levels.

In [18]:
# Query 9


query = """
SELECT country,
       date,
       value AS spread,
       ROW_NUMBER() OVER (
           PARTITION BY country
           ORDER BY value DESC
       ) AS spread_rank
FROM panel_data
WHERE variable = 'spread'
"""

pd.read_sql(query, conn)




,country,date,spread,spread_rank
0,AT,2012Q1,1.2119772005772005,1
1,AT,2011Q4,1.19130303030303,2
2,AT,2012Q2,1.1147983671299466,3
3,AT,2009Q1,1.0503340548340567,4
4,AT,2009Q2,0.8301212121212127,5
...,...,...,...,...
1195,PT,2004Q1,0.0851897860593506,96
1196,PT,2003Q1,0.0848412698412701,97
1197,PT,2004Q4,0.0784199134199128,98
1198,PT,2005Q2,0.0457215007215001,99


WHAT:
This query ranks spread observations within each country.

HOW:
It uses a window function (ROW_NUMBER) to assign a rank to
each observation based on the spread value.

SELECT country
Selects the country identifier.

date
Selects the quarter of the observation.

value AS spread
Retrieves the spread value and renames the column as "spread".

ROW_NUMBER() OVER (
Creates a ranking number for each observation using a window function.

PARTITION BY country
Resets the ranking separately for each country.

ORDER BY value DESC
Orders the spreads from highest to lowest so the highest spread
receives rank 1.

) AS spread_rank
 Names the ranking column "spread_rank".

FROM panel_data
Specifies the dataset being used.

WHERE variable = 'spread'
 Filters rows so only spread observations are ranked.

WHY:
This query helps identify the highest spread periods for each country,
 which may correspond to financial stress or economic crises.

In [19]:
# Query 10
# This query calculates a rolling average of spreads for each country over time.

query = """
SELECT country,
       date,
       value AS spread,
       AVG(value) OVER (
           PARTITION BY country
           ORDER BY date
           ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
       ) AS rolling_avg_spread
FROM panel_data
WHERE variable = 'spread'
"""

pd.read_sql(query, conn)



,country,date,spread,rolling_avg_spread
0,AT,2000Q1,0.2431204278812968,0.243120
1,AT,2000Q2,0.2899663299663269,0.266543
2,AT,2000Q3,0.3221471359558308,0.285078
3,AT,2000Q4,0.3105314878104375,0.291441
4,AT,2001Q1,0.3196303030303031,0.310569
...,...,...,...,...
1195,PT,2023Q4,0.7411164274322166,0.802901
1196,PT,2024Q1,0.7346832611832634,0.760464
1197,PT,2024Q2,0.6767467532467535,0.731991
1198,PT,2024Q3,0.6422896668548832,0.698709


WHAT:
This query calculates a rolling average of sovereign spreads
for each country over time.

HOW:
It uses a window function with a moving window of observations.

SELECT country
Selects the country identifier.

date
Selects the time period of the observation.

value AS spread
Retrieves the spread value and renames it as "spread".

AVG(value) OVER (
Calculates the average spread using a window function.

PARTITION BY country
Separates the calculations for each country.

ORDER BY date
 Orders observations chronologically.

ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
Defines a rolling window consisting of the current observation
and the three previous observations.

) AS rolling_avg_spread
 Names the calculated column "rolling_avg_spread".

FROM panel_data
 Specifies the dataset.

WHERE variable = 'spread'
Filters rows so only spread observations are used.

 WHY:
Rolling averages help smooth short-term fluctuations and
highlight longer-term trends in sovereign borrowing costs.

In [20]:
# Query 11
# This query joins spread and current account balance data.

query = """
SELECT s.country,
       s.date,
       s.value AS spread,
       c.value AS current_account
FROM panel_data s
JOIN panel_data c
ON s.country = c.country
AND s.date = c.date
WHERE s.variable = 'spread'
AND c.variable = 'ca_gdp'
LIMIT 50
"""

pd.read_sql(query, conn)



,country,date,spread,current_account
0,AT,2000Q1,0.2431204278812968,0.0136153510263074
1,AT,2000Q2,0.2899663299663269,-0.0211722310815999
2,AT,2000Q3,0.3221471359558308,-0.0108710849562721
3,AT,2000Q4,0.3105314878104375,-0.0051804901492667
4,AT,2001Q1,0.3196303030303031,0.0091088792791777
5,AT,2001Q2,0.3008623604465699,-0.0298220721003528
6,AT,2001Q3,0.2726259552042158,-0.0069722034195062
7,AT,2001Q4,0.2411083856467497,0.0001247680094823
8,AT,2002Q1,0.1993242424242431,0.0442397336977815
9,AT,2002Q2,0.1976115440115426,-0.0046923943211259


WHAT:
This query combines sovereign spreads with current account
balance data for the same country and quarter.

HOW:
The dataset is joined with itself to combine spread values
and current account values.

SELECT s.country
Selects the country identifier.

s.date
Selects the time period of the observation.

s.value AS spread
Retrieves spread values and renames the column as "spread".

c.value AS current_account
Retrieves current account values and renames the column
as "current_account".

FROM panel_data s
 Specifies the dataset and assigns alias "s" for spread observations.

JOIN panel_data c
Joins the dataset with itself using alias "c"
 to represent current account observations.

ON s.country = c.country AND s.date = c.date
Matches rows where country and date are the same.

WHERE s.variable = 'spread'
Filters rows so alias "s" only includes spread observations.

AND c.variable = 'ca_gdp'
Filters rows so alias "c" only includes current account values.

LIMIT 50
Restricts the output to the first 50 rows.

WHY:
This query allows analysis of whether a country's external
balance (current account) is related to its sovereign borrowing costs.

In [22]:
# Query 12


query = """
SELECT country,
       AVG(value) AS avg_spread
FROM panel_data
WHERE variable = 'spread'
GROUP BY country

UNION ALL

SELECT 'ALL_COUNTRIES' AS country,
       AVG(value) AS avg_spread
FROM panel_data
WHERE variable = 'spread'
"""

pd.read_sql(query, conn)



,country,avg_spread
0,AT,0.348723
1,BE,0.488914
2,DE,0.000000
3,ES,1.026203
4,FI,0.275042
5,FR,0.363014
6,GR,3.924514
7,IE,1.070474
8,IT,1.336504
9,LU,0.100724


WHAT:
This query calculates the average sovereign spread for each country
and also calculates the overall average spread across all countries.

HOW:
Because SQLite does not support the ROLLUP operator, this query
simulates a rollup using UNION ALL.
The first SELECT calculates average spreads by country.
The second SELECT calculates the overall average spread for the dataset.

SELECT country
Returns the country name for each group.

AVG(value) AS avg_spread
Calculates the average spread for each country.

FROM panel_data
Specifies the dataset being used.

 WHERE variable = 'spread'
Filters rows so only sovereign spread observations are included.

GROUP BY country
Groups rows by country so the average spread is calculated for each country.

UNION ALL
Combines the country-level results with the overall dataset result.

SELECT 'ALL_COUNTRIES' AS country
Creates a label for the overall summary row.

AVG(value) AS avg_spread
Calculates the average spread across the entire dataset.

FROM panel_data
Specifies the dataset again for the total calculation.

 WHERE variable = 'spread'
Ensures only spread observations are used for the overall average.

 WHY:
This query produces both country-level averages and the overall dataset
 average in one result table, which replicates the behavior of a rollup.


In [23]:
# Query 13


query = """
SELECT country,
       AVG(value) AS avg_gdp_growth
FROM panel_data
WHERE variable = 'gdp_growth'
GROUP BY country
ORDER BY avg_gdp_growth DESC
"""

pd.read_sql(query, conn)



,country,avg_gdp_growth
0,IE,1.242390
1,LU,0.553625
2,ES,0.408339
3,BE,0.385948
4,NL,0.375239
5,AT,0.331959
6,FR,0.309947
7,FI,0.277044
8,DE,0.248734
9,PT,0.239222


WHAT:
This query calculates the average GDP growth rate for each country.

HOW:
The query filters the dataset to only include rows where the variable
equals 'gdp_growth'. It then calculates the average value using AVG()
and groups the results by country.

SELECT country
Returns the country identifier.

AVG(value) AS avg_gdp_growth
Calculates the average GDP growth rate for each country.

 FROM panel_data
Specifies the dataset used for the analysis.

WHERE variable = 'gdp_growth'
Filters rows so only GDP growth observations are included.

GROUP BY country
Groups rows by country so the average GDP growth is calculated
separately for each country.

ORDER BY avg_gdp_growth DESC
Sorts countries from highest to lowest average GDP growth.

WHY:
This query helps compare economic growth performance across countries.

In [24]:
# Query 14

query = """
SELECT s.country,
       s.date,
       s.value AS spread,
       g.value AS gdp_growth
FROM panel_data s
JOIN panel_data g
ON s.country = g.country AND s.date = g.date
WHERE s.variable = 'spread'
AND g.variable = 'gdp_growth'
AND g.value < 0
"""

pd.read_sql(query, conn)




,country,date,spread,gdp_growth
0,AT,2001Q1,0.3196303030303031,-0.30145287373084
1,AT,2001Q2,0.3008623604465699,-0.28603936750446
2,AT,2002Q3,0.1877417152895431,-0.01338575834246
3,AT,2002Q4,0.1361041177823754,-0.49962967290007
4,AT,2007Q4,0.1289726807719127,-0.09028737600846
...,...,...,...,...
327,PT,2013Q3,5.110417529330573,-0.2820291264937299
328,PT,2014Q1,3.252339826839827,-0.71991450479647
329,PT,2020Q1,0.8803333333333333,-4.34794183534901
330,PT,2020Q2,1.2367619047619047,-16.30489338642426


WHAT:
This query identifies sovereign spreads during periods when GDP growth is negative.

HOW:
The dataset is joined with itself to combine spread observations with
GDP growth observations. The query then filters rows where GDP growth
is below zero.

SELECT s.country
Selects the country identifier.

s.date
Selects the time period.

s.value AS spread
Retrieves the sovereign spread value.

g.value AS gdp_growth
Retrieves the GDP growth value.

FROM panel_data s
Defines the dataset and assigns alias "s" for spread observations.

JOIN panel_data g
Joins the dataset with itself to obtain GDP growth values.

ON s.country = g.country AND s.date = g.date
Matches rows from the same country and quarter.

WHERE s.variable = 'spread'
Filters rows so alias "s" contains only spread observations.

AND g.variable = 'gdp_growth'
Filters rows so alias "g" contains only GDP growth observations.

AND g.value < 0
Keeps only observations where GDP growth is negative.

WHY:
This query helps analyze how borrowing costs behave during economic downturns.

In [25]:
# Query 15

query = """
SELECT country,
       date,
       value AS debt_gdp,
       ROW_NUMBER() OVER (
           PARTITION BY country
           ORDER BY value DESC
       ) AS debt_rank
FROM panel_data
WHERE variable = 'debt_gdp'
"""

pd.read_sql(query, conn)



,country,date,debt_gdp,debt_rank
0,AT,2021Q1,87.0,1
1,AT,2015Q3,86.8,2
2,AT,2015Q2,86.7,3
3,AT,2016Q1,86.2,4
4,AT,2021Q2,85.9,5
...,...,...,...,...
1195,PT,2022Q4,111.2,96
1196,PT,2023Q1,110.8,97
1197,PT,2023Q2,108.2,98
1198,PT,2023Q3,105.4,99


WHAT:
This query identifies the highest debt-to-GDP observation for each country.

HOW:
A window function ranks debt observations within each country.
The highest debt value receives rank 1.

SELECT country
Returns the country identifier.

date
Returns the time period.

value AS debt_gdp
Retrieves the debt-to-GDP value.

ROW_NUMBER() OVER (
Creates a ranking for each observation.

PARTITION BY country
Resets the ranking separately for each country.

ORDER BY value DESC
Orders the values from highest to lowest.

) AS debt_rank
Assigns a ranking number to each observation.

FROM panel_data
 Specifies the dataset used.

 WHERE variable = 'debt_gdp'
 Filters rows so only debt-to-GDP observations are included.

WHY:
This query helps identify the time period when each country
experienced its highest government debt level.


In [26]:
# Query 16

query = """
SELECT s.country,
       s.date,
       s.value AS spread,
       r.value AS reer_change
FROM panel_data s
JOIN panel_data r
ON s.country = r.country AND s.date = r.date
WHERE s.variable = 'spread'
AND r.variable = 'reer_change'
"""

pd.read_sql(query, conn)



,country,date,spread,reer_change
0,AT,2000Q1,0.2431204278812968,n.a.
1,AT,2000Q2,0.2899663299663269,-0.0079612322602973
2,AT,2000Q3,0.3221471359558308,-0.0058269364968599
3,AT,2000Q4,0.3105314878104375,-0.0043870424314744
4,AT,2001Q1,0.3196303030303031,0.0175902425267908
...,...,...,...,...
1195,PT,2023Q4,0.7411164274322166,-0.0010171905197844
1196,PT,2024Q1,0.7346832611832634,-0.0028510334996434
1197,PT,2024Q2,0.6767467532467535,0.0086115933149526
1198,PT,2024Q3,0.6422896668548832,-0.0054333153347732


WHAT:
This query combines sovereign spreads with real exchange rate changes.

HOW:
The dataset is joined with itself, so spread values and exchange rate
changes appear in the same row for the same country and quarter.

SELECT s.country
Returns the country identifier.

s.date
Returns the time period.

s.value AS spread
Retrieves sovereign spread values.

r.value AS reer_change
Retrieves real exchange rate change values.

FROM panel_data s
Defines the dataset and assigns the alias "s" for spread observations.

JOIN panel_data r
Joins the dataset with itself to obtain exchange rate changes.

 ON s.country = r.country AND s.date = r.date
Matches rows with the same country and quarter.

 WHERE s.variable = 'spread'
Filters rows so alias "s" only includes spread observations.

AND r.variable = 'reer_change'
Filters rows so alias "r" only includes exchange rate change observations.

 WHY:
Exchange rate movements can influence financial stability and investor
confidence, which may affect sovereign borrowing costs.